In [1]:
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import torch
import json
from torch.utils.data import Dataset, DataLoader

PAD_CALL = 16  # pitch_call ids are 0-15
PAD_TYPE = 18  # pitch_type ids are 0-17
N_PITCH_TYPES = 18

STRIKE_CALLS = {0, 2, 8, 9, 11, 15}
FOUL_CALLS = {4, 5, 6, 12}
BALL_CALLS = {1, 3, 7, 10}

EVAL_PITCHER = 693433  # Bryan Woo — per-pitcher val slice
EVAL_BATTER = 643289
DATA_ROOT = Path("/home/matthew/Documents/PitchPredict/data")


def count_from_calls(calls):
    balls, strikes = 0, 0
    for c in calls:
        if c in BALL_CALLS:
            balls = min(3, balls + 1)
        elif c in FOUL_CALLS:
            if strikes < 2:
                strikes += 1
        elif c in STRIKE_CALLS:
            strikes = min(2, strikes + 1)
    return balls, strikes


def parse_pitch_file(path):
    with open(path) as file:
        item = json.load(file)
    calls = item["pitch_calls_so_far"]
    types = item["pitch_types_so_far"]
    balls, strikes = count_from_calls(calls)
    label = int(item["pitch_type"])
    if label < 0 or label >= N_PITCH_TYPES:
        label = 13  # UN
    numeric = [
        item["batter_avg"], item["batter_obp"], item["batter_slg"],
        item["offense_score"], item["defense_score"],
        item["inning"], item["at_bat_number"], item["pitch_number"],
    ]
    context = [
        item["outs"], item["on_1b"], item["on_2b"], item["on_3b"],
        item["inning_half"], item["p_throws"], item["stand"], balls, strikes
    ]
    last_call = calls[-1] if calls else PAD_CALL
    last_type = types[-1] if types else PAD_TYPE
    last_call2 = calls[-2] if len(calls) > 1 else PAD_CALL
    last_type2 = types[-2] if len(types) > 1 else PAD_TYPE
    last_call3 = calls[-3] if len(calls) > 2 else PAD_CALL
    last_type3 = types[-3] if len(types) > 2 else PAD_TYPE
    return numeric, context, last_call, last_type, last_call2, last_type2, last_call3, last_type3, item["pitcher_id"], item["batter_id"], label


class PitchesDataset(Dataset):
    def __init__(self, root, pitcher_id=None, cache_path=None):
        root = Path(root)
        cache_path = Path(cache_path) if cache_path else root.parent / "cache" / f"{root.name}.pt"
        if cache_path.exists():
            blob = torch.load(cache_path, weights_only=True)
            self._set_from_blob(blob)
            print(f"{root.name}: {len(self)} pitches (cache)")
        else:
            self._load_json(root)
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(self._blob(), cache_path)
            print(f"{root.name}: {len(self)} pitches (cached to {cache_path})")
        if pitcher_id is not None:
            self._keep(self.pitcher_mlbam == pitcher_id)

    def _load_json(self, root):
        paths = list(root.glob("*.json"))
        numeric, context, last_call, last_type, last_call2, last_type2, last_call3, last_type3, pitcher_mlbam, batter_mlbam, labels = (
            [], [], [], [], [], [], [], [], [], [], []
        )
        with ThreadPoolExecutor(max_workers=8) as pool:
            for i, row in enumerate(pool.map(parse_pitch_file, paths, chunksize=256)):
                if i and i % 50000 == 0:
                    print(f"  loaded {i}/{len(paths)}")
                n, c, lc, lt, lc2, lt2, lc3, lt3, pmlbam, bmlbam, label = row
                numeric.append(n)
                context.append(c)
                last_call.append(lc)
                last_type.append(lt)
                last_call2.append(lc2)
                last_type2.append(lt2)
                last_call3.append(lc3)
                last_type3.append(lt3)
                pitcher_mlbam.append(pmlbam)
                batter_mlbam.append(bmlbam)
                labels.append(label)
        self.numeric = torch.tensor(numeric, dtype=torch.float32)
        self.context = torch.tensor(context, dtype=torch.long)
        self.last_call = torch.tensor(last_call, dtype=torch.long)
        self.last_type = torch.tensor(last_type, dtype=torch.long)
        self.last_call2 = torch.tensor(last_call2, dtype=torch.long)
        self.last_type2 = torch.tensor(last_type2, dtype=torch.long)
        self.last_call3 = torch.tensor(last_call3, dtype=torch.long)
        self.last_type3 = torch.tensor(last_type3, dtype=torch.long)
        self.pitcher_mlbam = torch.tensor(pitcher_mlbam, dtype=torch.long)
        self.pitcher_idx = torch.zeros(len(labels), dtype=torch.long)
        self.batter_mlbam = torch.tensor(batter_mlbam, dtype=torch.long)
        self.batter_idx = torch.zeros(len(labels), dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def _blob(self):
        return {
            "numeric": self.numeric,
            "context": self.context,
            "last_call": self.last_call,
            "last_type": self.last_type,
            "last_call2": self.last_call2,
            "last_type2": self.last_type2,
            "last_call3": self.last_call3,
            "last_type3": self.last_type3,
            "pitcher_mlbam": self.pitcher_mlbam,
            "batter_mlbam": self.batter_mlbam,
            "labels": self.labels,
        }

    def _set_from_blob(self, blob):
        self.numeric = blob["numeric"]
        self.context = blob["context"]
        self.last_call = blob["last_call"]
        self.last_type = blob["last_type"]
        self.last_call2 = blob["last_call2"]
        self.last_type2 = blob["last_type2"]
        self.last_call3 = blob["last_call3"]
        self.last_type3 = blob["last_type3"]
        self.pitcher_mlbam = blob["pitcher_mlbam"]
        self.batter_mlbam = blob["batter_mlbam"]
        self.labels = blob["labels"]
        self.pitcher_idx = torch.zeros(len(self.labels), dtype=torch.long)
        self.batter_idx = torch.zeros(len(self.labels), dtype=torch.long)

    def _keep(self, mask):
        self.numeric = self.numeric[mask]
        self.context = self.context[mask]
        self.last_call = self.last_call[mask]
        self.last_type = self.last_type[mask]
        self.last_call2 = self.last_call2[mask]
        self.last_type2 = self.last_type2[mask]
        self.last_call3 = self.last_call3[mask]
        self.last_type3 = self.last_type3[mask]
        self.pitcher_mlbam = self.pitcher_mlbam[mask]
        self.pitcher_idx = self.pitcher_idx[mask]
        self.batter_mlbam = self.batter_mlbam[mask]
        self.batter_idx = self.batter_idx[mask]
        self.labels = self.labels[mask]

    def numeric_matrix(self):
        return self.numeric

    def standardize(self, mean, std):
        self.numeric = (self.numeric - mean) / std.clamp_min(1e-6)

    def assign_pitcher_idx(self, pitcher_to_idx, unk):
        hi = max(int(self.pitcher_mlbam.max()), max(pitcher_to_idx))
        table = torch.full((hi + 1,), unk, dtype=torch.long)
        for mlbam, i in pitcher_to_idx.items():
            table[mlbam] = i
        self.pitcher_idx = table[self.pitcher_mlbam]

    def assign_batter_idx(self, batter_to_idx, unk):
        hi = max(int(self.batter_mlbam.max()), max(batter_to_idx))
        table = torch.full((hi + 1,), unk, dtype=torch.long)
        for mlbam, i in batter_to_idx.items():
            table[mlbam] = i
        self.batter_idx = table[self.batter_mlbam]


    def subset_pitcher(self, mlbam):
        mask = self.pitcher_mlbam == mlbam
        sub = object.__new__(PitchesDataset)
        sub.numeric = self.numeric[mask]
        sub.context = self.context[mask]
        sub.last_call = self.last_call[mask]
        sub.last_type = self.last_type[mask]
        sub.last_call2 = self.last_call2[mask]
        sub.last_type2 = self.last_type2[mask]
        sub.last_call3 = self.last_call3[mask]
        sub.last_type3 = self.last_type3[mask]
        sub.pitcher_mlbam = self.pitcher_mlbam[mask]
        sub.pitcher_idx = self.pitcher_idx[mask]
        sub.batter_idx = self.batter_idx[mask]
        sub.labels = self.labels[mask]
        return sub

    def subset_change(self):
        mask = (self.last_type + self.last_type2 + self.last_type3) / 3 != self.last_type
        sub = object.__new__(PitchesDataset)
        sub.numeric = self.numeric[mask]
        sub.context = self.context[mask]
        sub.last_call = self.last_call[mask]
        sub.last_type = self.last_type[mask]
        sub.last_call2 = self.last_call2[mask]
        sub.last_type2 = self.last_type2[mask]
        sub.last_call3 = self.last_call3[mask]
        sub.last_type3 = self.last_type3[mask]
        sub.pitcher_mlbam = self.pitcher_mlbam[mask]
        sub.batter_mlbam = self.batter_mlbam[mask]
        sub.pitcher_idx = self.pitcher_idx[mask]
        sub.batter_idx = self.batter_idx[mask]
        sub.labels = self.labels[mask]
        return sub

    def __len__(self):
        return int(self.labels.shape[0])

    def __getitem__(self, idx):
        return {
            "numeric": self.numeric[idx],
            "context": self.context[idx],
            "last_call": self.last_call[idx],
            "last_type": self.last_type[idx],
            "last_call2": self.last_call2[idx],
            "last_type2": self.last_type2[idx],
            "last_call3": self.last_call3[idx],
            "last_type3": self.last_type3[idx],
            "pitcher_idx": self.pitcher_idx[idx],
            "batter_idx": self.batter_idx[idx]
        }, self.labels[idx]


In [2]:
print("Loading all pitchers (JSON once, then cache)...")
train = PitchesDataset(DATA_ROOT / "train")
val = PitchesDataset(DATA_ROOT / "val")

pitcher_ids = sorted(set(train.pitcher_mlbam.tolist()))
pitcher_to_idx = {mlbam: i for i, mlbam in enumerate(pitcher_ids)}
unk_pitcher = len(pitcher_to_idx)
n_pitchers = unk_pitcher + 1
n_classes = N_PITCH_TYPES

train.assign_pitcher_idx(pitcher_to_idx, unk_pitcher)
val.assign_pitcher_idx(pitcher_to_idx, unk_pitcher)

batter_ids = sorted(set(train.batter_mlbam.tolist()))
batter_to_idx = {mlbam: i for i, mlbam in enumerate(batter_ids)}
unk_batter = len(batter_to_idx)
n_batters = unk_batter + 1

train.assign_batter_idx(batter_to_idx, unk_batter)
val.assign_batter_idx(batter_to_idx, unk_batter)

label_counts = Counter(train.labels.tolist())
majority_id, majority_n = label_counts.most_common(1)[0]
print(f"Pitchers: {len(pitcher_ids)}  (+1 unk)   train pitches: {len(train)}")
print(f"League majority class {majority_id}: {100 * majority_n / len(train):.1f}%")
league_top2 = sum(n for _, n in label_counts.most_common(2))
print(f"League majority top-2: {100 * league_top2 / len(train):.1f}%")

woo_train = train.subset_pitcher(EVAL_PITCHER)
if len(woo_train):
    woo_counts = Counter(woo_train.labels.tolist())
    w_id, w_n = woo_counts.most_common(1)[0]
    print(f"Woo train pitches: {len(woo_train)}  majority {w_id}: {100 * w_n / len(woo_train):.1f}%")
    woo_top2 = sum(n for _, n in woo_counts.most_common(2))
    print(f"Woo majority top-2: {100 * woo_top2 / len(woo_train):.1f}%")


mean = train.numeric.mean(dim=0)
std = train.numeric.std(dim=0)
train.standardize(mean, std)
val.standardize(mean, std)

woo_val = val.subset_pitcher(EVAL_PITCHER)
print(f"Woo val pitches: {len(woo_val)}")


Loading all pitchers (JSON once, then cache)...
  loaded 50000/883358
  loaded 100000/883358
  loaded 150000/883358
  loaded 200000/883358
  loaded 250000/883358
  loaded 300000/883358
  loaded 350000/883358
  loaded 400000/883358
  loaded 450000/883358
  loaded 500000/883358
  loaded 550000/883358
  loaded 600000/883358
  loaded 650000/883358
  loaded 700000/883358
  loaded 750000/883358
  loaded 800000/883358
  loaded 850000/883358
train: 883358 pitches (cached to /home/matthew/Documents/PitchPredict/data/cache/train.pt)
  loaded 50000/126584
  loaded 100000/126584
val: 126584 pitches (cached to /home/matthew/Documents/PitchPredict/data/cache/val.pt)
Pitchers: 1086  (+1 unk)   train pitches: 883358
League majority class 2: 31.2%
League majority top-2: 47.1%
Woo train pitches: 3357  majority 2: 47.3%
Woo majority top-2: 70.4%
Woo val pitches: 473


In [3]:
print(len(train), "train  |  ", len(val), "val  |  ", len(woo_val), "woo val")
print(train[0])

883358 train  |   126584 val  |   473 woo val
({'numeric': tensor([ 0.1959,  0.1257, -0.0818,  0.7058, -0.8566,  0.0160, -0.0698,  0.6243]), 'context': tensor([1, 0, 0, 0, 1, 0, 1, 1, 2]), 'last_call': tensor(1), 'last_type': tensor(9), 'last_call2': tensor(4), 'last_type2': tensor(2), 'last_call3': tensor(0), 'last_type3': tensor(2), 'pitcher_idx': tensor(445), 'batter_idx': tensor(262)}, tensor(3))


In [4]:
import os
import torch
from torch import nn


device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


class PitchModel(nn.Module):
    def __init__(self, n_classes, n_pitchers, n_batters):
        super().__init__()

        self.pitcher_embed = nn.Embedding(num_embeddings=n_pitchers, embedding_dim=16)
        self.batter_embed = nn.Embedding(num_embeddings=n_batters, embedding_dim=16)
        self.p_throws_embed = nn.Embedding(num_embeddings=2, embedding_dim=2)
        self.half_embed = nn.Embedding(num_embeddings=2, embedding_dim=2)
        self.outs_embed = nn.Embedding(num_embeddings=4, embedding_dim=2)
        self.on_1b_embed = nn.Embedding(num_embeddings=2, embedding_dim=2)
        self.on_2b_embed = nn.Embedding(num_embeddings=2, embedding_dim=2)
        self.on_3b_embed = nn.Embedding(num_embeddings=2, embedding_dim=2)
        self.stand_embed = nn.Embedding(num_embeddings=2, embedding_dim=2)


        self.balls_embed = nn.Embedding(num_embeddings=4, embedding_dim=8)
        self.strikes_embed = nn.Embedding(num_embeddings=3, embedding_dim=8)


        self.last_call_embed = nn.Embedding(num_embeddings=17, embedding_dim=4, padding_idx=PAD_CALL)
        self.last_type_embed = nn.Embedding(num_embeddings=19, embedding_dim=4, padding_idx=PAD_TYPE)

        self.last_call2_embed = nn.Embedding(num_embeddings=17, embedding_dim=4, padding_idx=PAD_CALL)
        self.last_type2_embed = nn.Embedding(num_embeddings=19, embedding_dim=4, padding_idx=PAD_TYPE)

        self.last_call3_embed = nn.Embedding(num_embeddings=17, embedding_dim=4, padding_idx=PAD_CALL)
        self.last_type3_embed = nn.Embedding(num_embeddings=19, embedding_dim=4, padding_idx=PAD_TYPE)

        # 10 numeric + pitcher 16 + 7 context*2 + last_call 4 + last_type 4
        total_dims = 8 + 16 +16 + 2 * 7 + 16 + 16

        # LSTM processes merged step features: (4 call dims + 4 type dims = 8 inputs per step)
        self.seq_lstm = nn.LSTM(
            input_size=8, hidden_size=16, num_layers=1, batch_first=True
        )

        self.dropout = nn.Dropout(0.2)

        self.reduce_dims = nn.Linear(total_dims, 64)
        self.linear_relu_stack = nn.Sequential(
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.output_dims = nn.Linear(32, n_classes)

    def forward(self, x):
        emb_pitcher = self.dropout(self.pitcher_embed(x["pitcher_idx"]))
        emb_batter = self.dropout(self.batter_embed(x["batter_idx"]))
        emb_outs = self.outs_embed(x["context"][:, 0])
        emb_on_1b = self.on_1b_embed(x["context"][:, 1])
        emb_on_2b = self.on_2b_embed(x["context"][:, 2])
        emb_on_3b = self.on_3b_embed(x["context"][:, 3])
        emb_half = self.half_embed(x["context"][:, 4])
        emb_throws = self.p_throws_embed(x["context"][:, 5])
        emb_stand = self.stand_embed(x["context"][:, 6])

        emb_balls = self.balls_embed(x["context"][:, 7]) * 3
        emb_strikes = self.strikes_embed(x["context"][:, 8]) * 3

        emb_last_call = self.last_call_embed(x["last_call"])
        emb_last_type = self.last_type_embed(x["last_type"])
        emb_last_call2 = self.last_call2_embed(x["last_call2"])
        emb_last_type2 = self.last_type2_embed(x["last_type2"])
        emb_last_call3 = self.last_call3_embed(x["last_call3"])
        emb_last_type3 = self.last_type3_embed(x["last_type3"])

        seq_calls = torch.stack([emb_last_call3, emb_last_call2, emb_last_call], dim=1)
        seq_types = torch.stack([emb_last_type3, emb_last_type2, emb_last_type], dim=1)
        seq_input = torch.cat([seq_calls, seq_types], dim=2)
        _, (hn, _) = self.seq_lstm(seq_input)
        emb_sequence_summary = hn[-1]

        x = torch.cat([
            x["numeric"],
            emb_pitcher,
            emb_batter,
            emb_throws,
            emb_outs,
            emb_on_1b,
            emb_on_2b,
            emb_on_3b,
            emb_half,
            emb_stand,
            emb_balls,
            emb_strikes,
            emb_sequence_summary
        ], dim=1)
        x = self.reduce_dims(x)
        x = self.linear_relu_stack(x)
        return self.output_dims(x)

Using cpu device


In [5]:
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
model = PitchModel(n_classes, n_pitchers, n_batters)
batch_size = 256


def iter_batches(ds, batch_size, shuffle=False):
    n = len(ds)
    order = torch.randperm(n) if shuffle else torch.arange(n)
    for start in range(0, n, batch_size):
        sl = order[start:start + batch_size]
        yield {
            "numeric": ds.numeric[sl],
            "context": ds.context[sl],
            "last_call": ds.last_call[sl],
            "last_type": ds.last_type[sl],
            "last_call2": ds.last_call2[sl],
            "last_type2": ds.last_type2[sl],
            "last_call3": ds.last_call3[sl],
            "last_type3": ds.last_type3[sl],
            "pitcher_idx": ds.pitcher_idx[sl],
            "batter_idx": ds.batter_idx[sl]
        }, ds.labels[sl]


def topk_hits(pred, y, k=2):
    top = pred.topk(k, dim=1).indices
    hit1 = (top[:, 0] == y).sum().item()
    hit2 = (top == y.unsqueeze(1)).any(dim=1).sum().item()
    return hit1, hit2


def train_loop(dataset, model, loss_fn, optimizer):
    size = len(dataset)
    num_batches = max(1, (size + batch_size - 1) // batch_size)
    model.train()
    train_loss, hit1, hit2 = 0, 0, 0
    for batch, (X, y) in enumerate(iter_batches(dataset, batch_size, shuffle=True)):
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()

        train_loss += loss.item()
        b1, b2 = topk_hits(pred, y, k=3)
        hit1 += b1
        hit2 += b2

        if batch % 500 == 0:
            current = batch * batch_size + len(y)
            print(f"loss: {loss.item():>7f}  [{current:>5d}/{size:>5d}]")

    train_loss /= num_batches
    print(
        f"Train: top-1 {(100*hit1/size):>0.1f}%, top-3 {(100*hit2/size):>0.1f}%, "
        f"Avg loss: {train_loss:>8f}"
    )


def test_loop(dataset, model, loss_fn, split="Val"):
    size = len(dataset)
    if size == 0:
        print(f"{split}: empty")
        return
    num_batches = max(1, (size + batch_size - 1) // batch_size)
    model.eval()
    test_loss, hit1, hit2 = 0, 0, 0

    with torch.no_grad():
        for X, y in iter_batches(dataset, batch_size, shuffle=False):
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            b1, b2 = topk_hits(pred, y, k=3)
            hit1 += b1
            hit2 += b2

    test_loss /= num_batches
    print(
        f"{split}: top-1 {(100*hit1/size):>0.1f}%, top-3 {(100*hit2/size):>0.1f}%, "
        f"Avg loss: {test_loss:>8f}"
    )


In [6]:


loss_fn = nn.CrossEntropyLoss()


optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train, model, loss_fn, optimizer)
    test_loop(val, model, loss_fn, split="Val (all)")
    test_loop(woo_val, model, loss_fn, split="Val (Woo)")
    print()
print("Done!")

Epoch 1
-------------------------------
loss: 3.007934  [  256/883358]
loss: 1.831741  [128256/883358]
loss: 1.794480  [256256/883358]
loss: 1.638929  [384256/883358]
loss: 1.598758  [512256/883358]
loss: 1.606160  [640256/883358]
loss: 1.577972  [768256/883358]
Train: top-1 37.9%, top-3 74.5%, Avg loss: 1.679630
Val (all): top-1 41.6%, top-3 82.6%, Avg loss: 1.467585
Val (Woo): top-1 49.5%, top-3 87.1%, Avg loss: 1.333991

Epoch 2
-------------------------------
loss: 1.559714  [  256/883358]
loss: 1.396332  [128256/883358]
loss: 1.471331  [256256/883358]
loss: 1.434548  [384256/883358]
loss: 1.451071  [512256/883358]
loss: 1.494826  [640256/883358]
loss: 1.413219  [768256/883358]
Train: top-1 41.9%, top-3 82.9%, Avg loss: 1.447781
Val (all): top-1 43.1%, top-3 85.4%, Avg loss: 1.369338
Val (Woo): top-1 46.7%, top-3 86.7%, Avg loss: 1.310669

Epoch 3
-------------------------------
loss: 1.372270  [  256/883358]
loss: 1.387942  [128256/883358]
loss: 1.496570  [256256/883358]
loss: 1.3

In [7]:
vocabs = json.loads((DATA_ROOT / "meta" / "vocabs.json").read_text())
TYPE_TO_ID = vocabs["pitch_type"]
CALL_TO_ID = vocabs["pitch_call"]
ID_TO_TYPE = {int(v): k for k, v in TYPE_TO_ID.items()}


def predict_situation(situation):
    """situation uses human field names; numeric fields are standardized with train mean/std."""
    calls = [CALL_TO_ID[c] for c in situation.get("calls_so_far", [])]
    types = [TYPE_TO_ID[t] for t in situation.get("types_so_far", [])]
    balls, strikes = count_from_calls(calls)
    mlbam = situation.get("pitcher_id", EVAL_PITCHER)
    bmlbam = situation.get("batter_id", EVAL_BATTER)
    pitcher_idx = pitcher_to_idx.get(mlbam, unk_pitcher)
    batter_idx = batter_to_idx.get(bmlbam, unk_batter)

    numeric = torch.tensor([
        situation["batter_avg"],
        situation["batter_obp"],
        situation["batter_slg"],
        situation["offense_score"],
        situation["defense_score"],
        situation["inning"],
        situation["at_bat_number"],
        situation["pitch_number"]
    ], dtype=torch.float32)
    numeric = (numeric - mean) / std.clamp_min(1e-6)

    batch = {
        "numeric": numeric.unsqueeze(0),
        "context": torch.tensor([[
            situation["outs"],
            situation["on_1b"],
            situation["on_2b"],
            situation["on_3b"],
            situation["inning_half"],  # 0=Top, 1=Bot
            situation["p_throws"],     # 0=R, 1=L
            situation["stand"],        # 0=R, 1=L
            balls,
            strikes,
        ]], dtype=torch.long),
        "last_call": torch.tensor([calls[-1] if calls else PAD_CALL], dtype=torch.long),
        "last_type": torch.tensor([types[-1] if types else PAD_TYPE], dtype=torch.long),
        "last_call2": torch.tensor([calls[-2] if len(calls) > 1 else PAD_CALL], dtype=torch.long),
        "last_type2": torch.tensor([types[-2] if len(types) > 1 else PAD_TYPE], dtype=torch.long),
        "last_call3": torch.tensor([calls[-3] if len(calls) > 2 else PAD_CALL], dtype=torch.long),
        "last_type3": torch.tensor([types[-3] if len(types) > 2 else PAD_TYPE], dtype=torch.long),
        "pitcher_idx": torch.tensor([pitcher_idx], dtype=torch.long),
        "batter_idx": torch.tensor([batter_idx], dtype=torch.long),
    }

    model.eval()
    with torch.no_grad():
        logits = model(batch)
        probs = torch.softmax(logits, dim=-1)[0]

    ranked = sorted(
        ((ID_TO_TYPE.get(i, str(i)), float(probs[i])) for i in range(len(probs))),
        key=lambda x: -x[1],
    )
    last_type_name = ID_TO_TYPE.get(types[-1], "none") if types else "none"
    id_to_call = {int(v): k for k, v in CALL_TO_ID.items()}
    last_call_name = id_to_call.get(calls[-1], "none") if calls else "none"
    print(f"Count: {balls}-{strikes}  |  last: {last_type_name} / {last_call_name}")
    print(f"Best guess: {ranked[0][0]}  ({100 * ranked[0][1]:.1f}%)")
    print("Likelihoods:")
    for name, p in ranked:
        if p < 0.005:
            continue
        print(f"  {name:4s}  {100 * p:5.1f}%")
    return ranked


# 1-2 putaway after three fastballs: pitcher ahead, batter protecting.
example = {"game_date":20250928,"at_bat_number":76,"pitch_number":5,"pitcher_id":686826,"batter_id":669200,"batter_avg":0.204,"batter_obp":0.278,"batter_slg":0.245,"pitch_calls_so_far":[1],"pitch_types_so_far":[2],"outs":2,"on_1b":1,"on_2b":1,"on_3b":0,"offense_score":12,"defense_score":2,"inning":8,"inning_half":1,"p_throws":0,"stand":0,"pitch_type":1,"outcome_type":0}

predict_situation(example)

Count: 0-0  |  last: none / none
Best guess: FF  (28.9%)
Likelihoods:
  FF     28.9%
  SL     27.6%
  SI     24.9%
  FC      8.2%
  CH      6.5%
  CU      2.8%


[('FF', 0.2888490557670593),
 ('SL', 0.2763858139514923),
 ('SI', 0.24947355687618256),
 ('FC', 0.08229804784059525),
 ('CH', 0.06453218311071396),
 ('CU', 0.02769956924021244),
 ('UN', 0.004759255796670914),
 ('FA', 0.003967755939811468),
 ('CS', 0.0009134197607636452),
 ('PO', 0.0003995713486801833),
 ('SV', 0.0003424311871640384),
 ('ST', 0.0001867808896349743),
 ('EP', 0.0001649920886848122),
 ('FS', 2.6546522349235602e-05),
 ('KN', 8.721960398361261e-07),
 ('KC', 1.966976412859367e-07),
 ('FO', 3.953739835527159e-10),
 ('SC', 6.167245300771785e-16)]

In [8]:
# Define the file path
model_path = "pitch_predictor.pth"

# Save the model's state dictionary
torch.save(model.state_dict(), model_path)

print(f"Model saved successfully to {model_path}")

Model saved successfully to pitch_predictor.pth
